# Running the eval

Three small functions run the whole dataset through the prompt:

- `run_prompt(test_case)` merges the test case into the prompt and calls Claude
- `run_test_case(test_case)` runs the prompt and grades the output
- `run_eval(dataset)` calls `run_test_case` for every case and collects the results

Grading is a placeholder here (`score = 10`), real graders come in the next lessons.

## Setup

The client reads `ANTHROPIC_API_KEY` (and `ANTHROPIC_BASE_URL`, if set) from `.env` in the repo root.

In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helpers

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    # The API rejects system=None, so optional params are only sent when set
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return "".join(block.text for block in message.content if block.type == "text")

## Eval pipeline

In [3]:
import json


def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [4]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }

In [5]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

## Run it

In [6]:
with open("dataset.json") as f:
    dataset = json.load(f)

In [7]:
results = run_eval(dataset)

for result in results:
    print(f"[{result['score']}] {len(result['output'])} chars | {result['test_case']['task']}")

[10] 2808 chars | Write a Python function that extracts the AWS account ID from an ARN string
[10] 1517 chars | Create a JSON object that represents an AWS IAM policy allowing read-only access to S3 bucket objects
[10] 2614 chars | Write a regex pattern that validates AWS S3 bucket names according to the naming rules (3-63 characters, lowercase letters, numbers, hyphens, no consecutive hyphens, cannot start/end with hyphen)


The placeholder gives every answer a 10, even though the answers are long explanations
instead of plain code. That gap is exactly what a real grader has to catch.

In [8]:
print(results[0]['output'])

# AWS Account ID Extractor from ARN

Here's a Python function that extracts the AWS account ID from an ARN string:

```python
def extract_aws_account_id(arn: str) -> str:
    """
    Extract the AWS account ID from an ARN string.
    
    ARN format: arn:partition:service:region:account-id:resource-type/resource-id
    
    Args:
        arn (str): The ARN string to parse
        
    Returns:
        str: The AWS account ID (12 digits), or empty string if not found
        
    Raises:
        ValueError: If the ARN format is invalid
    """
    if not isinstance(arn, str):
        raise ValueError("ARN must be a string")
    
    if not arn.startswith("arn:"):
        raise ValueError("Invalid ARN: must start with 'arn:'")
    
    parts = arn.split(":")
    
    if len(parts) < 6:
        raise ValueError("Invalid ARN: insufficient number of components")
    
    account_id = parts[4]
    
    # Account ID should be 12 digits or empty (for some resources)
    if account_id and not a